# Install & Setup
Run this cell first to install document-graph and sync notebooks.

In [ ]:
!aws s3 sync s3://graphrag-artifacts-705909755305/document-graph-notebooks/ ~/SageMaker/document-graph/ --quiet
!pip install -q graphrag-lexical-graph falkordb
!pip install -q --force-reinstall ~/SageMaker/document-graph/wheels/document_graph-3.0.0-py3-none-any.whl
print('✅ Installed')

## Verify

In [ ]:
from document_graph import Node, Edge, __version__
from document_graph.graph_build import node_to_cypher, edge_to_cypher
from document_graph.query import DocumentGraphQueryEngine
print(f'document-graph v{__version__} ✅')

## Test Neptune Connection

In [ ]:
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory

GRAPH_STORE = 'neptune-db://obs-app-dev-graph.cluster-czupj1ab2tc6.us-east-1.neptune.amazonaws.com:8182'
gs = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()
result = gs.query('MATCH (n) RETURN count(n) as cnt LIMIT 1')
print(f'Neptune connected ✅ — {result}')

## Write a Test Node

In [ ]:
node = Node(id='sagemaker-test-1', labels=['TestNode'], properties={'source': 'sagemaker', 'status': 'ok'})
cypher, params = node_to_cypher(node, tenant_id='test')
print(f'Cypher: {cypher}')
gs.query(cypher, params)
print('✅ Node written to Neptune')

# Read back
engine = DocumentGraphQueryEngine(gs, tenant_id='test')
result = engine.find_by_property('TestNode', 'source', 'sagemaker')
print(f'Query result: {result}')